# QSAR Model Analyzer and Predictor

This notebook provides a single interface for loading trained QSAR models and generating predictions for new compounds

In [1]:
import pandas as pd
import numpy as np
import joblib
import os
from pathlib import Path
from sklearn.preprocessing import StandardScaler
import subprocess as sp
import time
import warnings
warnings.filterwarnings('ignore')

# Input library for feature extraction, and feature engineering
from rdkit import Chem
from rdkit.Chem import Descriptors, rdFingerprintGenerator

home_dir = os.path.dirname(os.path.abspath('__file__'))

In [2]:
# Define completed training model files.
model_notebooks = [
    '01_LinearRegression.ipynb'
    #Add more model files here
    #'02_XGBoost.ipynb',
]

for notebook in model_notebooks:
    notebook_path = Path(home_dir + f"\Models\{notebook}")
   
    try:
        print(f"Training the Model - {notebook}")
        result = sp.run([
            'jupyter', 'nbconvert',
            '--to', 'notebook',
            '--execute',
            '--stdout',  # Output to stdout instead of file
            str(notebook_path)
        ], capture_output=True, text=True, cwd=home_dir + f"\Models")
        print(f"Model - {notebook} Training completed")

    except Exception as e:
        print(f"Error executing {notebook}: {e}")

Training the Model - 01_LinearRegression.ipynb
Model - 01_LinearRegression.ipynb Training completed


## Load models

Trained models and its parameters are saved in Libary. This block loads from the folder

In [3]:
# Define model names which are already developed
model_names = ['01_LinearRegression']

# Initialize storage for model components
models = {}
scalers = {}
feature_infos = {}

# Load all model artifacts
loaded_models = []

for model_name in model_names:
    # Load model
    model_path = home_dir + f"\Library\{model_name}_model.pkl"
    models[model_name] = joblib.load(model_path)
    
    # Load scaler
    scaler_path = home_dir + f"\Library\{model_name}_feature_scaler.pkl"
    scalers[model_name] = joblib.load(scaler_path)
    
    # Load feature information
    feature_path = home_dir + f"\Library\{model_name}_features.pkl"
    feature_infos[model_name] = joblib.load(feature_path)

    loaded_models.append(model_name)

print(f"\nSuccessfully loaded {len(loaded_models)} models: {loaded_models}")


Successfully loaded 1 models: ['01_LinearRegression']


## Performance comparision and chosing best model

In [4]:
# Read performance data for each model
performance_data = []

df = pd.read_csv(home_dir + f'\Results\Model_performance_comparison.csv')
df = df.sort_values('R2_CV_Mean', ascending=False)
df['Rank'] = range(1, len(df) + 1)
print(df[['Model', 'Features', 'Feature_Selection', 'R2_CV_Mean', 'RMSE_Mean', 'Rank']].to_string(index=False))

           Model    Features  Feature_Selection  R2_CV_Mean  RMSE_Mean  Rank
LinearRegression Descriptors Variance Threshold      0.4893     0.4157     1
LinearRegression Descriptors Variance Threshold      0.4893     0.4157     2
LinearRegression Descriptors Variance Threshold      0.4893     0.4157     3
LinearRegression Descriptors Variance Threshold      0.4893     0.4157     4
LinearRegression Descriptors Variance Threshold      0.4893     0.4157     5
LinearRegression Descriptors Variance Threshold      0.4893     0.4157     6


## Read virtual compounds and pre-process

In [5]:
in_file = home_dir + f'\Virtual\\test.csv'

df_input = pd.read_csv(in_file)
print(f"Input shape and Info: {df_input.shape}")

df_input = df_input.dropna()
#print(f"Removing empty columns:")
#print(f"{df_input.shape}")
#print(f"{df_input.info()}")

# Define descriptor names
L1_descriptor_names = ['L1_MolWt', 'L1_MolLogP', 'L1_NumHDonors', 'L1_NumHAcceptors', 'L1_TPSA', 'L1_NumRotatableBonds', 'L1_NumAromaticRings', 'L1_NumAliphaticRings', 'L1_FractionCSP3', 'L1_HeavyAtomCount']
L2_descriptor_names = ['L2_MolWt', 'L2_MolLogP', 'L2_NumHDonors', 'L2_NumHAcceptors', 'L2_TPSA', 'L2_NumRotatableBonds', 'L2_NumAromaticRings', 'L2_NumAliphaticRings', 'L2_FractionCSP3', 'L2_HeavyAtomCount']
L3_descriptor_names = ['L3_MolWt', 'L3_MolLogP', 'L3_NumHDonors', 'L3_NumHAcceptors', 'L3_TPSA', 'L3_NumRotatableBonds', 'L3_NumAromaticRings', 'L3_NumAliphaticRings', 'L3_FractionCSP3', 'L3_HeavyAtomCount']

def extract_descriptors(smiles):
  mol = Chem.MolFromSmiles(smiles)
  if mol:
    descriptors = [
      Descriptors.MolWt(mol),
      Descriptors.MolLogP(mol),
      Descriptors.NumHDonors(mol),
      Descriptors.NumHAcceptors(mol),
      Descriptors.TPSA(mol),
      Descriptors.NumRotatableBonds(mol),
      Descriptors.NumAromaticRings(mol),
      Descriptors.NumAliphaticRings(mol),
      Descriptors.FractionCSP3(mol),
      Descriptors.HeavyAtomCount(mol),
    ]
    return descriptors
  else:
    return None

# Define ligand types and their corresponding descriptor names
feature_configs = [
    ('L1', L1_descriptor_names),
    ('L2', L2_descriptor_names), 
    ('L3', L3_descriptor_names)
]

for feature, descriptor_names in feature_configs:
    #print(f"Extracting descriptors for {feature}...")
    
    # Initialize descriptor dictionary
    in_descriptors = {name: [] for name in descriptor_names}
    
    # Extract descriptors for each SMILES in this ligand column
    for smiles in df_input[feature]:
        descriptors_values = extract_descriptors(smiles)
        if descriptors_values:
            for i, name in enumerate(descriptor_names):
                in_descriptors[name].append(descriptors_values[i])
        else:
            # Handle invalid SMILES
            for name in descriptor_names:
                in_descriptors[name].append(np.nan)
    
    # Add extracted descriptors to dataframe
    for name in descriptor_names:
        df_input[name] = in_descriptors[name]

#Again do data cleaning - remove empty values:
df_input = df_input.dropna()
print("Post desriptor extraction and clean-up: ", df_input.shape)


Input shape and Info: (19, 3)
Post desriptor extraction and clean-up:  (18, 33)


[23:21:11] Can't kekulize mol.  Unkekulized atoms: 1 2 4
